In [ ]:
import os
import sys
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
from PIL import Image
import requests

try:
    from ultralytics import YOLO
    ULTRALYTICS_OK = True
except Exception:
    ULTRALYTICS_OK = False
    print("⚠️ Ultralytics not available")

try:
    from transformers import CLIPProcessor, CLIPModel
    TRANSFORMERS_OK = True
except Exception:
    TRANSFORMERS_OK = False
    print("⚠️ Transformers not available")

try:
    from filterpy.kalman import KalmanFilter
    from scipy.optimize import linear_sum_assignment
    import scipy
    ADVANCED_TRACKING_AVAILABLE = True
except Exception:
    print("⚠️ Advanced tracking not available. Using basic tracking.")
    ADVANCED_TRACKING_AVAILABLE = False

def setup_geometric_reid():
    try:
        print("🔄 Setting up geometric ReID (XFeat+LightGlue+SuperPoint)...")
        try:
            from lightglue import LightGlue, SuperPoint
            from lightglue.utils import rbd
        except ImportError:
            print("📦 Installing geometric ReID packages...")
            os.system('pip install git+https://github.com/cvg/LightGlue.git --quiet')
            os.system('pip install git+https://github.com/verlab/accelerated_features.git --quiet')
            os.system('pip install kornia --quiet')
            from lightglue import LightGlue, SuperPoint
            from lightglue.utils import rbd

        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        superpoint_model = SuperPoint(max_num_keypoints=512).eval().to(device)
        lightglue_model = LightGlue(features='superpoint').eval().to(device)
        print("✅ Geometric ReID models loaded successfully!")
        return superpoint_model, lightglue_model, True
    except Exception as e:
        print(f"⚠️ Geometric ReID setup failed: {e}")
        return None, None, False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
superpoint_model, lightglue_model, GEOMETRIC_REID_AVAILABLE = setup_geometric_reid()

def iou_batch(bb_test, bb_gt):
    bb_gt = np.expand_dims(bb_gt, 0)
    bb_test = np.expand_dims(bb_test, 1)
    xx1 = np.maximum(bb_test[..., 0], bb_gt[..., 0])
    yy1 = np.maximum(bb_test[..., 1], bb_gt[..., 1])
    xx2 = np.minimum(bb_test[..., 2], bb_gt[..., 2])
    yy2 = np.minimum(bb_test[..., 3], bb_gt[..., 3])
    w = np.maximum(0., xx2 - xx1)
    h = np.maximum(0., yy2 - yy1)
    wh = w * h
    o = wh / ((bb_test[..., 2] - bb_test[..., 0]) * (bb_test[..., 3] - bb_test[..., 1])
              + (bb_gt[..., 2] - bb_gt[..., 0]) * (bb_gt[..., 3] - bb_gt[..., 1]) - wh + 1e-6)
    return o

def compute_iou_vector(tracker_bbox, detections):
    if len(detections) == 0:
        return []
    return iou_batch(detections[:, :4], tracker_bbox).flatten()

# Enhanced occlusion detection from the original algorithm
def detect_occlusion_enhanced(iou_vector, primary_iou_threshold=0.25, secondary_iou_threshold=0.12):
    """
    Enhanced occlusion detection that identifies multiple overlapping objects
    and different types of occlusion scenarios.
    """
    if not isinstance(iou_vector, (list, np.ndarray)) or len(iou_vector) <= 1:
        return {
            'is_occluded': False,
            'occlusion_type': 'none',
            'occluding_objects': 0,
            'confidence': 1.0
        }

    # Convert to numpy array if it's a list
    iou_array = np.array(iou_vector) if isinstance(iou_vector, list) else iou_vector

    # Sort IoU scores in descending order
    sorted_iou = np.sort(iou_array)[::-1]
    best_iou = sorted_iou[0]
    second_best_iou = sorted_iou[1] if len(sorted_iou) > 1 else 0

    # Count objects with significant overlap
    significant_overlaps = np.sum(iou_array >= secondary_iou_threshold)
    moderate_overlaps = np.sum(iou_array >= 0.05)

    occlusion_info = {
        'is_occluded': False,
        'occlusion_type': 'none',
        'occluding_objects': int(significant_overlaps - 1),  # Exclude primary match
        'confidence': float(best_iou)
    }

    # Different occlusion scenarios
    if significant_overlaps > 1:
        if best_iou > primary_iou_threshold and second_best_iou > secondary_iou_threshold:
            occlusion_info.update({
                'is_occluded': True,
                'occlusion_type': 'partial_overlap',  # Target partially overlapped by another object
                'severity': float(min(second_best_iou / max(best_iou, 1e-6), 1.0))
            })
        elif best_iou < primary_iou_threshold and significant_overlaps >= 2:
            occlusion_info.update({
                'is_occluded': True,
                'occlusion_type': 'heavy_occlusion',  # Target heavily occluded
                'severity': float(1.0 - best_iou)
            })
        elif moderate_overlaps > 2:
            occlusion_info.update({
                'is_occluded': True,
                'occlusion_type': 'crowd_occlusion',  # Target in crowded area
                'severity': float(moderate_overlaps / len(iou_array))
            })

    return occlusion_info

def extract_geometric_features(crop):
    if not GEOMETRIC_REID_AVAILABLE:
        return None
    try:
        if crop.shape[0] < 32 or crop.shape[1] < 32:
            crop = cv2.resize(crop, (64, 64))
        crop_gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if len(crop.shape) == 3 else crop
        tensor = torch.from_numpy(crop_gray).float().to(DEVICE).unsqueeze(0).unsqueeze(0) / 255.0
        with torch.no_grad():
            features = superpoint_model({'image': tensor})
        return {
            'keypoints': features['keypoints'][0],
            'descriptors': features['descriptors'][0],
            'scores': features.get('scores', [None])[0]
        }
    except Exception:
        return None

def compute_geometric_similarity(feat1, feat2):
    if not GEOMETRIC_REID_AVAILABLE or feat1 is None or feat2 is None:
        return 0.0
    try:
        data = {
            'image0': {k: v.unsqueeze(0) for k, v in feat1.items()},
            'image1': {k: v.unsqueeze(0) for k, v in feat2.items()}
        }
        with torch.no_grad():
            matches = lightglue_model(data)
        if 'matches0' in matches:
            valid_matches = matches['matches0'][0][matches['matches0'][0] >= 0]
            num_matches = len(valid_matches)
            total_kpts = min(len(feat1['keypoints']), len(feat2['keypoints']))
            if total_kpts > 0:
                return min(1.0, num_matches / max(total_kpts, 1) * 2.5)
        return 0.0
    except Exception:
        return 0.0

def compute_geometric_similarity_with_gallery(det_feat, geom_gallery):
    if det_feat is None or not geom_gallery:
        return 0.0
    similarities = [compute_geometric_similarity(det_feat, g_feat) for g_feat in geom_gallery]
    return max(similarities) if similarities else 0.0

class EnhancedUltimateKalmanTracker:
    count = 0
    def __init__(self, bbox, now_frame_idx=0):
        self.id = EnhancedUltimateKalmanTracker.count
        EnhancedUltimateKalmanTracker.count += 1

        if ADVANCED_TRACKING_AVAILABLE:
            self.kf = KalmanFilter(dim_x=7, dim_z=4)
            self.kf.F = np.array([[1,0,0,0,1,0,0],[0,1,0,0,0,1,0],[0,0,1,0,0,0,1],
                                  [0,0,0,1,0,0,0],[0,0,0,0,1,0,0],[0,0,0,0,0,1,0],
                                  [0,0,0,0,0,0,1]])
            self.kf.H = np.array([[1,0,0,0,0,0,0],[0,1,0,0,0,0,0],
                                  [0,0,1,0,0,0,0],[0,0,0,1,0,0,0]])
            self.kf.R[2:,2:] *= 10.
            self.kf.P[4:,4:] *= 1000.
            self.kf.P *= 10.
            self.kf.Q[-1,-1] *= 0.01
            self.kf.Q[4:,4:] *= 0.01
            self.kf.x[:4] = self.convert_bbox_to_z(bbox)
        else:
            self.position = bbox.copy()

        # Basic tracking states
        self.time_since_update = 0
        self.hits = 0
        self.hit_streak = 0
        self.age = 0
        self.clip_feature_gallery = deque(maxlen=8)
        self.geometric_feature_gallery = deque(maxlen=6)
        self.tracking_state = 'ACTIVE'
        self.lost_age = 0
        self.max_lost_age = 90
        self.last_seen_frame_idx = now_frame_idx

        # Enhanced occlusion handling attributes
        self.occlusion_history = deque(maxlen=15)  # Track occlusion patterns
        self.occlusion_confidence = 0.0
        self.occlusion_recovery_attempts = 0
        self.max_occlusion_recovery_attempts = 3
        self.consecutive_occlusion_frames = 0
        self.max_consecutive_occlusion_frames = 8

        # Enhanced ReID verification
        self.reid_verification_history = deque(maxlen=10)
        self.false_positive_count = 0
        self.track_hijack_protection = True
        self.identity_confidence = 1.0
        self.identity_stability_threshold = 0.7

        # Occlusion-aware feature management
        self.pre_occlusion_features = {
            'clip': None,
            'geometric': None,
            'bbox': None,
            'frame_idx': -1
        }

        # Enhanced motion prediction
        self.velocity_history = deque(maxlen=12)
        self.trajectory = deque(maxlen=40)
        self.last_position = None
        self.motion_consistency_score = 1.0

        # Recovery and exit detection
        self.frames_since_good_match = 0
        self.max_frames_without_good_match = 20  # Reduced for better responsiveness
        self.last_recovered_frame = -1
        self.recovery_cooldown = 6  # Reduced for faster recovery
        self.exit_confirmation_frames = 0
        self.exit_confirmation_threshold = 3  # Frames to confirm exit

        # Quality metrics
        self.average_confidence = 1.0
        self.confidence_scores = deque(maxlen=7)
        self.match_quality_history = deque(maxlen=10)

        # Text description and appearance - CRITICAL FOR PREVENTING HIJACKING
        self.text_desc_index = -1
        self.appearance_stability = 1.0

        # YoloReID-style Identity Protection
        self.identity_lock = False  # Lock identity once established
        self.identity_establishment_frames = 5  # Frames needed to establish identity
        self.established_identity_features = {
            'clip': deque(maxlen=5),  # Store established identity features
            'geometric': deque(maxlen=5),
            'text_similarity_history': deque(maxlen=10)
        }

        # Anti-hijacking mechanisms
        self.original_text_similarity = 0.0  # Original text match score
        self.hijack_resistance_threshold = 0.6  # Threshold to resist hijacking
        self.non_target_rejection_count = 0  # Count rejections of non-target entities
        self.max_non_target_rejections = 3  # Max rejections before stricter verification

    def handle_occlusion_scenario(self, occlusion_info, current_bbox, clip_feature, geometric_feature, frame_idx):
        """
        Enhanced occlusion handling based on the original algorithm's logic
        """
        self.occlusion_history.append(occlusion_info)

        if occlusion_info['is_occluded']:
            self.consecutive_occlusion_frames += 1

            # Store pre-occlusion features for later verification
            if self.consecutive_occlusion_frames == 1:  # First frame of occlusion
                self.pre_occlusion_features = {
                    'clip': self.clip_feature_gallery[-1] if self.clip_feature_gallery else None,
                    'geometric': self.geometric_feature_gallery[-1] if self.geometric_feature_gallery else None,
                    'bbox': self.get_state()[0].copy(),
                    'frame_idx': frame_idx - 1
                }

            # Handle different occlusion types
            if occlusion_info['occlusion_type'] == 'heavy_occlusion':
                # For heavy occlusion, rely more on motion prediction
                self.occlusion_confidence = max(0.0, self.occlusion_confidence - 0.1)
                return self._handle_heavy_occlusion(current_bbox, clip_feature, geometric_feature)

            elif occlusion_info['occlusion_type'] == 'partial_overlap':
                # For partial overlap, use enhanced ReID verification
                return self._handle_partial_occlusion(current_bbox, clip_feature, geometric_feature, occlusion_info)

            elif occlusion_info['occlusion_type'] == 'crowd_occlusion':
                # In crowd situations, be more conservative with updates
                return self._handle_crowd_occlusion(current_bbox, clip_feature, geometric_feature, occlusion_info)
        else:
            # No occlusion - normal tracking
            if self.consecutive_occlusion_frames > 0:
                # Just recovered from occlusion
                recovery_success = self._verify_occlusion_recovery(clip_feature, geometric_feature)
                if recovery_success:
                    self.consecutive_occlusion_frames = 0
                    self.occlusion_confidence = min(1.0, self.occlusion_confidence + 0.2)
                    return 'clean_recovery'
                else:
                    # Possible track hijacking - be cautious
                    return 'potential_hijack'
            else:
                self.consecutive_occlusion_frames = 0
                self.occlusion_confidence = min(1.0, self.occlusion_confidence + 0.1)
                return 'normal_tracking'

    def _handle_heavy_occlusion(self, current_bbox, clip_feature, geometric_feature):
        """Handle heavy occlusion scenarios"""
        if self.consecutive_occlusion_frames > self.max_consecutive_occlusion_frames:
            self.tracking_state = 'LOST'
            return 'occlusion_lost'

        # Use motion prediction primarily
        predicted_pos = self.get_predicted_chw_smoothed()
        motion_consistency = self._check_motion_consistency(current_bbox, predicted_pos)

        if motion_consistency < 0.3:  # Motion doesn't match prediction
            return 'occlusion_drift'

        # Limited feature update during heavy occlusion
        if clip_feature is not None and len(self.clip_feature_gallery) > 0:
            clip_sim = self.compute_clip_similarity_with_gallery(clip_feature, list(self.clip_feature_gallery))
            if clip_sim < 0.4:  # Low similarity suggests wrong target
                return 'occlusion_mismatch'

        return 'occlusion_tracking'

    def _handle_partial_occlusion(self, current_bbox, clip_feature, geometric_feature, occlusion_info):
        """Handle partial occlusion with enhanced verification"""
        # Use both CLIP and geometric features for verification
        clip_verified = False
        geometric_verified = False

        if clip_feature is not None and len(self.clip_feature_gallery) > 0:
            clip_sim = self.compute_clip_similarity_with_gallery(clip_feature, list(self.clip_feature_gallery))
            clip_verified = clip_sim > 0.5  # Higher threshold during partial occlusion

        if geometric_feature is not None and len(self.geometric_feature_gallery) > 0:
            geom_sim = compute_geometric_similarity_with_gallery(geometric_feature, list(self.geometric_feature_gallery))
            geometric_verified = geom_sim > 0.3

        # Multi-modal verification
        verification_score = 0
        if clip_verified: verification_score += 0.6
        if geometric_verified: verification_score += 0.4

        # Consider occlusion severity
        severity_factor = 1.0 - occlusion_info.get('severity', 0.5)
        final_verification = verification_score * severity_factor

        if final_verification > 0.4:
            return 'partial_occlusion_verified'
        else:
            return 'partial_occlusion_rejected'

    def _handle_crowd_occlusion(self, current_bbox, clip_feature, geometric_feature, occlusion_info):
        """Handle crowd occlusion scenarios"""
        # In crowded scenarios, be more conservative
        if clip_feature is not None and len(self.clip_feature_gallery) > 2:
            # Compare with multiple recent features
            recent_features = list(self.clip_feature_gallery)[-3:]
            similarities = [torch.cosine_similarity(clip_feature.unsqueeze(0), feat.unsqueeze(0)).item()
                           for feat in recent_features if clip_feature is not None]
            if similarities:  # Check if we have any similarities
                avg_similarity = np.mean(similarities)
                if avg_similarity < 0.6:  # Stricter threshold in crowd
                    return 'crowd_identity_uncertain'

        # Check for consistent motion in crowd
        if len(self.trajectory) > 5:
            recent_trajectory = list(self.trajectory)[-5:]
            motion_variance = np.var([pos[0] for pos in recent_trajectory]) + np.var([pos[1] for pos in recent_trajectory])

            if motion_variance > 1000:  # Erratic motion suggests confusion
                return 'crowd_motion_inconsistent'

        return 'crowd_tracking'

    def _verify_occlusion_recovery(self, clip_feature, geometric_feature):
        """Verify that we've correctly recovered from occlusion"""
        # Fixed boolean check for tensors
        clip_available = self.pre_occlusion_features['clip'] is not None
        geometric_available = self.pre_occlusion_features['geometric'] is not None

        if not clip_available and not geometric_available:
            return True  # No pre-occlusion features to compare

        recovery_confidence = 0.0

        # Compare with pre-occlusion CLIP features
        if clip_feature is not None and clip_available:
            clip_sim = torch.cosine_similarity(
                clip_feature.unsqueeze(0),
                self.pre_occlusion_features['clip'].unsqueeze(0)
            ).item()
            recovery_confidence += 0.6 * clip_sim

        # Compare with pre-occlusion geometric features
        if geometric_feature is not None and geometric_available:
            geom_sim = compute_geometric_similarity(geometric_feature, self.pre_occlusion_features['geometric'])
            recovery_confidence += 0.4 * geom_sim

        return recovery_confidence > 0.5

    def verify_identity_against_hijacking(self, clip_feature, text_features, text_descriptions, similarity_threshold):
        """
        YoloReID-style identity verification to prevent track hijacking
        """
        if self.text_desc_index < 0 or not self.identity_lock:
            return True, 1.0, "identity_not_established"

        # Check if the new detection still matches the original text description
        if clip_feature is not None and text_features is not None:
            current_text_similarities = (clip_feature @ text_features.T).squeeze()
            if current_text_similarities.dim() == 0:
                current_text_similarities = current_text_similarities.unsqueeze(0)

            # Get similarity to the tracker's original text description
            original_desc_similarity = current_text_similarities[self.text_desc_index].item()

            # Get best matching description (could be different from original)
            best_similarity, best_idx = current_text_similarities.max(), current_text_similarities.argmax()

            # Store text similarity history
            self.established_identity_features['text_similarity_history'].append(original_desc_similarity)

            # Check for potential hijacking scenarios
            hijacking_detected = False
            hijacking_reason = ""

            # Scenario 1: Current detection doesn't match original description well
            if original_desc_similarity < similarity_threshold * 0.7:  # More lenient than original threshold
                hijacking_detected = True
                hijacking_reason = f"original_desc_mismatch_{original_desc_similarity:.3f}"

            # Scenario 2: Current detection matches a different description much better
            elif best_idx.item() != self.text_desc_index and best_similarity.item() > original_desc_similarity + 0.15:
                hijacking_detected = True
                hijacking_reason = f"different_desc_match_{text_descriptions[best_idx.item()]}"

            # Scenario 3: Consistent decline in text similarity
            if len(self.established_identity_features['text_similarity_history']) >= 5:
                recent_similarities = list(self.established_identity_features['text_similarity_history'])[-5:]
                if all(sim < similarity_threshold * 0.8 for sim in recent_similarities):
                    hijacking_detected = True
                    hijacking_reason = "consistent_desc_decline"

            # Additional verification using established identity features
            if hijacking_detected and len(self.established_identity_features['clip']) > 0:
                # Compare with established identity features
                identity_similarities = [
                    torch.cosine_similarity(clip_feature.unsqueeze(0), established_feat.unsqueeze(0)).item()
                    for established_feat in self.established_identity_features['clip']
                ]
                avg_identity_similarity = np.mean(identity_similarities)

                # If it still matches established identity well, it might be lighting/angle change
                if avg_identity_similarity > self.hijack_resistance_threshold:
                    hijacking_detected = False
                    hijacking_reason = "identity_features_match"
                    print(f"🛡️ ID:{self.id} Potential hijacking avoided by identity verification: {avg_identity_similarity:.3f}")

            if hijacking_detected:
                self.non_target_rejection_count += 1
                print(f"🚫 ID:{self.id} HIJACKING DETECTED: {hijacking_reason}")
                return False, original_desc_similarity, hijacking_reason
            else:
                self.non_target_rejection_count = max(0, self.non_target_rejection_count - 1)
                return True, original_desc_similarity, "identity_verified"

        return True, 1.0, "no_text_features"

    def establish_identity(self, clip_feature, geometric_feature, text_similarity):
        """
        Establish and lock the tracker's identity after sufficient evidence
        """
        if not self.identity_lock:
            # Store features for identity establishment
            if clip_feature is not None:
                self.established_identity_features['clip'].append(clip_feature.clone())
            if geometric_feature is not None:
                self.established_identity_features['geometric'].append(geometric_feature)

            self.original_text_similarity = max(self.original_text_similarity, text_similarity)

            # Lock identity after sufficient frames
            if self.hits >= self.identity_establishment_frames:
                self.identity_lock = True
                print(f"🔒 ID:{self.id} Identity LOCKED to '{self.get_description()}' after {self.hits} frames")
        else:
            # Update established identity features (slowly)
            if clip_feature is not None and len(self.established_identity_features['clip']) > 0:
                # Only update if very similar to existing features (prevent drift)
                similarities = [
                    torch.cosine_similarity(clip_feature.unsqueeze(0), est_feat.unsqueeze(0)).item()
                    for est_feat in self.established_identity_features['clip']
                ]
                if max(similarities) > 0.8:  # Very high threshold for updates
                    self.established_identity_features['clip'].append(clip_feature.clone())

    def get_description(self, text_descriptions=None):
        """Get the text description for this tracker"""
        if text_descriptions and self.text_desc_index >= 0 and self.text_desc_index < len(text_descriptions):
            return text_descriptions[self.text_desc_index]
        return f"target_{self.text_desc_index}" if self.text_desc_index >= 0 else "unknown"

    def _check_motion_consistency(self, current_bbox, predicted_pos):
        """Check if current detection matches predicted motion"""
        if predicted_pos is None:
            return 1.0

        cx, cy, w, h = self._bbox_to_chw(current_bbox)
        pred_cx, pred_cy, pred_w, pred_h = predicted_pos

        # Position consistency
        pos_diff = np.sqrt((cx - pred_cx)**2 + (cy - pred_cy)**2)
        size_diff = abs(w - pred_w) + abs(h - pred_h)

        # Normalize by bbox size
        normalized_pos_diff = pos_diff / max(w, h, 1.0)
        normalized_size_diff = size_diff / max(w, h, 1.0)

        consistency = max(0.0, 1.0 - 0.5 * normalized_pos_diff - 0.3 * normalized_size_diff)
        return consistency

    def compute_clip_similarity_with_gallery(self, det_feat, clip_gallery):
        """Compute similarity with CLIP feature gallery"""
        if det_feat is None or not clip_gallery:
            return 0.0
        sims = torch.cosine_similarity(det_feat.unsqueeze(0), torch.stack(clip_gallery))
        return torch.max(sims).item()

    def update(self, bbox, clip_feature=None, geometric_feature=None, confidence=1.0,
               iou_vector=None, match_quality=1.0, now_frame_idx=0, text_features=None,
               text_descriptions=None, similarity_threshold=0.3):
        was_lost = (self.tracking_state == 'LOST')

        # Enhanced occlusion analysis
        if iou_vector is not None and len(iou_vector) > 0:
            occlusion_info = detect_occlusion_enhanced(iou_vector)
        else:
            occlusion_info = {'is_occluded': False}

        # YoloReID-style Identity Verification (CRITICAL for preventing hijacking)
        identity_verified = True
        identity_confidence = 1.0
        hijacking_reason = "none"

        if (self.identity_lock and text_features is not None and
            text_descriptions is not None and clip_feature is not None):
            identity_verified, identity_confidence, hijacking_reason = self.verify_identity_against_hijacking(
                clip_feature, text_features, text_descriptions, similarity_threshold
            )

            # Strict rejection for hijacking attempts
            if not identity_verified:
                print(f"🛡️ ID:{self.id} HIJACKING PREVENTED: {hijacking_reason}")
                self.false_positive_count += 1

                # Mark as lost if too many hijacking attempts
                if self.non_target_rejection_count >= self.max_non_target_rejections:
                    self.tracking_state = 'LOST'
                    print(f"🚫 ID:{self.id} Marked LOST due to repeated hijacking attempts")

                return  # Don't update at all

        # Handle occlusion scenario
        occlusion_result = self.handle_occlusion_scenario(
            occlusion_info, bbox, clip_feature, geometric_feature, now_frame_idx
        )

        # Decide whether to update based on occlusion analysis AND identity verification
        should_update = True
        update_features = True

        if occlusion_result in ['occlusion_mismatch', 'partial_occlusion_rejected',
                               'crowd_identity_uncertain', 'potential_hijack']:
            should_update = False
            update_features = False
            self.false_positive_count += 1
            print(f"🚫 ID:{self.id} Update rejected: {occlusion_result}")

        elif occlusion_result in ['occlusion_tracking', 'crowd_tracking']:
            should_update = True
            update_features = False  # Update position but not features

        elif occlusion_result in ['occlusion_drift', 'crowd_motion_inconsistent']:
            # Use motion prediction instead of detection
            if ADVANCED_TRACKING_AVAILABLE:
                predicted_bbox = self.convert_x_to_bbox(self.kf.x)[0]
            else:
                predicted_bbox = self.position
            bbox = predicted_bbox
            should_update = True
            update_features = False

        if should_update:
            # Standard update logic
            self.time_since_update = 0
            self.hits += 1
            self.hit_streak += 1
            self.lost_age = 0
            self.last_seen_frame_idx = now_frame_idx

            if ADVANCED_TRACKING_AVAILABLE:
                self.kf.update(self.convert_bbox_to_z(bbox))
            else:
                self.position = bbox.copy()

            # Update features only if safe to do so
            if update_features:
                if clip_feature is not None:
                    self.clip_feature_gallery.append(clip_feature.clone())
                if geometric_feature is not None:
                    self.geometric_feature_gallery.append(geometric_feature)

                # Establish/update identity (YoloReID-style)
                if text_features is not None and text_descriptions is not None:
                    current_text_similarity = identity_confidence if identity_verified else 0.0
                    self.establish_identity(clip_feature, geometric_feature, current_text_similarity)

            # Update tracking state
            if was_lost:
                self.tracking_state = 'RECOVERING'
                self.last_recovered_frame = now_frame_idx
                print(f"🔄 ID:{self.id} Recovered from occlusion! Result: {occlusion_result}")
            else:
                self.tracking_state = 'ACTIVE'

            # Update quality metrics
            self.confidence_scores.append(confidence)
            self.match_quality_history.append(match_quality)
            if len(self.confidence_scores) > 0:
                self.average_confidence = sum(self.confidence_scores) / len(self.confidence_scores)

            # Reset good match counter for good matches
            if match_quality > 0.35 and not occlusion_info['is_occluded']:
                self.frames_since_good_match = 0

            # Update identity confidence based on consistency AND hijacking resistance
            if len(self.reid_verification_history) > 0:
                recent_quality = list(self.match_quality_history)[-5:] if self.match_quality_history else [match_quality]
                base_confidence = min(1.0, np.mean(recent_quality) * 1.2)

                # Boost confidence if identity is verified and locked
                if identity_verified and self.identity_lock:
                    self.identity_confidence = min(1.0, base_confidence * 1.1)
                else:
                    self.identity_confidence = base_confidence * 0.9  # Reduce if not verified

    def predict(self):
        if ADVANCED_TRACKING_AVAILABLE:
            if (self.kf.x[6] + self.kf.x[2]) <= 0:
                self.kf.x[6] *= 0.0
            self.kf.predict()

        self.age += 1
        self.time_since_update += 1
        if self.time_since_update > 0:
            self.hit_streak = 0

        # Enhanced motion prediction
        pred_bbox = self.get_state().flatten()
        cx, cy, w, h = self._bbox_to_chw(pred_bbox)
        current_pos = (float(cx), float(cy), float(w), float(h))

        if self.last_position is not None:
            vx = current_pos[0] - self.last_position[0]
            vy = current_pos[1] - self.last_position[1]
            self.velocity_history.append((vx, vy))

        self.trajectory.append(current_pos)
        self.last_position = current_pos

        # Update motion consistency
        if len(self.velocity_history) > 3:
            velocities = list(self.velocity_history)[-3:]
            vel_variance = np.var([v[0] for v in velocities]) + np.var([v[1] for v in velocities])
            self.motion_consistency_score = max(0.0, 1.0 - vel_variance / 100.0)

        # State transitions
        if self.tracking_state == 'LOST':
            self.lost_age += 1
        if self.tracking_state == 'ACTIVE':
            self.frames_since_good_match += 1

        return self.get_state()

    def mark_lost(self, current_frame_idx):
        """Enhanced lost marking with occlusion awareness"""
        # Don't mark lost if we're handling occlusion
        if self.consecutive_occlusion_frames > 0 and self.consecutive_occlusion_frames < self.max_consecutive_occlusion_frames:
            return

        # Don't mark lost too quickly after recovery
        if (self.tracking_state == 'RECOVERING' and
            current_frame_idx - self.last_recovered_frame < self.recovery_cooldown):
            return

        # Enhanced exit confirmation
        self.exit_confirmation_frames += 1

        if (self.frames_since_good_match >= self.max_frames_without_good_match or
            self.exit_confirmation_frames >= self.exit_confirmation_threshold):
            self.tracking_state = 'LOST'
            self.occlusion_recovery_attempts = 0
            print(f"🚫 ID:{self.id} Marked as LOST (frames_without_match: {self.frames_since_good_match}, exit_frames: {self.exit_confirmation_frames})")

    def can_attempt_recovery(self, current_frame_idx):
        if self.tracking_state != 'LOST':
            return False
        if self.occlusion_recovery_attempts >= self.max_occlusion_recovery_attempts:
            return False
        return True

    def attempt_recovery(self, current_frame_idx):
        self.occlusion_recovery_attempts += 1
        print(f"🔄 ID:{self.id} Attempting recovery (attempt {self.occlusion_recovery_attempts}/{self.max_occlusion_recovery_attempts})")

    def should_delete(self):
        return self.lost_age > self.max_lost_age

    def get_state(self):
        if ADVANCED_TRACKING_AVAILABLE:
            return self.convert_x_to_bbox(self.kf.x)
        else:
            return self.position.reshape((1, 4))

    def get_clip_gallery(self):
        return list(self.clip_feature_gallery)

    def get_geometric_gallery(self):
        return list(self.geometric_feature_gallery)

    def get_predicted_chw_smoothed(self):
        if not self.trajectory:
            return self._bbox_to_chw(self.get_state().flatten())
        base_pos = self.trajectory[-1]
        if self.velocity_history and self.is_ghost():
            avg_vx = sum(v[0] for v in self.velocity_history) / len(self.velocity_history)
            avg_vy = sum(v[1] for v in self.velocity_history) / len(self.velocity_history)
            future_frames = min(self.lost_age, 12)
            projected_cx = base_pos[0] + avg_vx * future_frames * 0.7
            projected_cy = base_pos[1] + avg_vy * future_frames * 0.7
            return projected_cx, projected_cy, base_pos[2], base_pos[3]
        return base_pos

    def is_active(self):
        return self.tracking_state in ['ACTIVE', 'RECOVERING']

    def is_ghost(self):
        return self.tracking_state == 'LOST'

    def get_occlusion_status(self):
        """Get current occlusion status and details"""
        if len(self.occlusion_history) == 0:
            return "Clear"

        recent_occlusion = self.occlusion_history[-1]
        if not recent_occlusion['is_occluded']:
            return "Clear"

        occlusion_type = recent_occlusion['occlusion_type']
        severity = recent_occlusion.get('severity', 0.5)

        if occlusion_type == 'heavy_occlusion':
            return f"Heavy ({severity:.1f})"
        elif occlusion_type == 'partial_overlap':
            return f"Partial ({severity:.1f})"
        elif occlusion_type == 'crowd_occlusion':
            return f"Crowd ({severity:.1f})"
        else:
            return f"Occluded ({severity:.1f})"

    def _bbox_to_chw(self, bbox):
        w, h = max(1.0, bbox[2]-bbox[0]), max(1.0, bbox[3]-bbox[1])
        return bbox[0] + w/2, bbox[1] + h/2, w, h

    @staticmethod
    def convert_bbox_to_z(bbox):
        w, h = bbox[2]-bbox[0], bbox[3]-bbox[1]
        x, y = bbox[0]+w/2., bbox[1]+h/2.
        s, r = w*h, w/float(h) if h!=0 else 1
        return np.array([x, y, s, r]).reshape((4, 1))

    @staticmethod
    def convert_x_to_bbox(x):
        w = np.sqrt(abs(x[2]*x[3]))
        h = abs(x[2])/w if w > 1e-6 else 0
        return np.array([x[0]-w/2., x[1]-h/2., x[0]+w/2., x[1]+h/2.]).reshape((1, 4))

class EnhancedUltimateMultiModalTracker:
    def __init__(self, model_path='yolov8n.pt', clip_model_name="openai/clip-vit-base-patch32", device='auto'):
        self.device = 'cuda' if torch.cuda.is_available() and device == 'auto' else 'cpu'
        print(f"🔧 Using device: {self.device}")

        if not ULTRALYTICS_OK:
            raise RuntimeError("Ultralytics YOLO not available")
        if not TRANSFORMERS_OK:
            raise RuntimeError("Transformers not available")

        model_to_load = model_path if os.path.exists(model_path) else 'yolov8n.pt'
        self.model = YOLO(model_to_load).to(self.device)
        self.clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
        self.clip_model = CLIPModel.from_pretrained(clip_model_name).to(self.device)
        self.trackers = []

        # Enhanced thresholds for occlusion handling
        self.min_hits_for_display = 2
        self.iou_threshold = 0.25
        self.stage1_clip_threshold = 0.28  # Slightly higher for better precision
        self.stage1_geometric_threshold = 0.18
        self.reid_iou_threshold = 0.08
        self.stage2_clip_threshold = 0.25  # More lenient for recovery
        self.stage2_geometric_threshold = 0.22

        # Enhanced weights for multi-modal fusion
        self.stage1_iou_weight = 0.30  # Reduced IoU dependency
        self.stage1_clip_weight = 0.40  # Increased CLIP weight
        self.stage1_geometric_weight = 0.30 if GEOMETRIC_REID_AVAILABLE else 0.0
        self.stage2_clip_weight = 0.45
        self.stage2_geometric_weight = 0.55 if GEOMETRIC_REID_AVAILABLE else 0.0

        # Occlusion handling parameters
        self.track_hijack_prevention = True
        self.identity_verification_threshold = 0.6
        self.occlusion_recovery_boost = 0.15  # Boost similarity scores during recovery

        print(f"✅ ENHANCED ULTIMATE Multi-Modal Tracker initialized")
        print(f"   - CLIP: ✅")
        print(f"   - Geometric ReID: {'✅' if GEOMETRIC_REID_AVAILABLE else '❌'}")
        print(f"   - Advanced Occlusion Handling: ✅")
        print(f"   - Track Hijack Prevention: ✅")

    def extract_clip_features(self, crop):
        try:
            pil_image = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            inputs = self.clip_processor(images=[pil_image], return_tensors="pt").to(self.device)
            with torch.no_grad():
                features = self.clip_model.get_image_features(**inputs)
            return features / features.norm(dim=-1, keepdim=True)
        except:
            return None

    def compute_clip_similarity_with_gallery(self, det_feat, clip_gallery):
        if det_feat is None or not clip_gallery:
            return 0.0
        sims = torch.cosine_similarity(det_feat, torch.stack(clip_gallery))
        return torch.max(sims).item()

    def enhanced_associate(self, frame, detections, frame_idx):
        """
        Enhanced association with improved occlusion handling
        """
        if not self.trackers or len(detections) == 0:
            return np.empty((0,2), dtype=int), np.arange(len(detections)), list(range(len(self.trackers)))

        # Extract features for all detections
        det_clip_features = []
        det_geometric_features = []
        for det in detections:
            crop = frame[int(det[1]):int(det[3]), int(det[0]):int(det[2])]
            clip_feat = self.extract_clip_features(crop)
            det_clip_features.append(clip_feat[0] if clip_feat is not None else None)
            if GEOMETRIC_REID_AVAILABLE:
                geom_feat = extract_geometric_features(crop)
                det_geometric_features.append(geom_feat)
            else:
                det_geometric_features.append(None)

        active_trk_indices = [i for i, t in enumerate(self.trackers) if t.is_active()]
        lost_trk_indices = [i for i, t in enumerate(self.trackers) if t.is_ghost() and t.can_attempt_recovery(frame_idx)]
        matches = []
        unmatched_dets = list(range(len(detections)))

        # STAGE 1: Association with active trackers (enhanced with occlusion awareness)
        if active_trk_indices and unmatched_dets:
            active_trackers = [self.trackers[i] for i in active_trk_indices]
            det_bboxes = detections[unmatched_dets, :4]
            trk_bboxes = np.array([t.get_state()[0] for t in active_trackers])

            if len(det_bboxes) > 0 and len(trk_bboxes) > 0:
                iou_matrix = iou_batch(det_bboxes, trk_bboxes)
                cost_matrix = np.full((len(unmatched_dets), len(active_trackers)), 1e5)

                for i, det_idx in enumerate(unmatched_dets):
                    for j, trk in enumerate(active_trackers):
                        # Enhanced IoU gating with occlusion consideration
                        base_iou_threshold = 0.05
                        if trk.consecutive_occlusion_frames > 0:
                            base_iou_threshold = 0.02  # More lenient during occlusion

                        if iou_matrix[i, j] < base_iou_threshold:
                            continue

                        # Compute multi-modal similarities
                        clip_sim = self.compute_clip_similarity_with_gallery(
                            det_clip_features[det_idx], trk.get_clip_gallery())

                        geometric_sim = 0.0
                        if GEOMETRIC_REID_AVAILABLE:
                            geometric_sim = compute_geometric_similarity_with_gallery(
                                det_geometric_features[det_idx], trk.get_geometric_gallery())

                        # Enhanced similarity computation with occlusion boost
                        recovery_boost = 0.0
                        if trk.tracking_state == 'RECOVERING':
                            recovery_boost = self.occlusion_recovery_boost

                        # Apply occlusion-aware boost
                        if trk.consecutive_occlusion_frames > 0:
                            clip_sim += recovery_boost
                            geometric_sim += recovery_boost

                        # Compute costs
                        iou_cost = 1 - iou_matrix[i, j]
                        clip_cost = 1 - max(0, min(1.0, clip_sim))
                        geometric_cost = 1 - max(0, min(1.0, geometric_sim))

                        # Enhanced cost computation with track hijack prevention
                        if GEOMETRIC_REID_AVAILABLE:
                            base_cost = (self.stage1_iou_weight * iou_cost +
                                        self.stage1_clip_weight * clip_cost +
                                        self.stage1_geometric_weight * geometric_cost)
                        else:
                            base_cost = (0.5 * iou_cost + 0.5 * clip_cost)

                        # Apply track hijack prevention penalty
                        if self.track_hijack_prevention and trk.identity_confidence < self.identity_verification_threshold:
                            hijack_penalty = (1.0 - trk.identity_confidence) * 0.3
                            base_cost += hijack_penalty

                        cost_matrix[i, j] = base_cost

                # Hungarian assignment with enhanced cost matrix
                if ADVANCED_TRACKING_AVAILABLE:
                    rows, cols = linear_sum_assignment(cost_matrix)
                    for r, c in zip(rows, cols):
                        # Enhanced acceptance criteria
                        acceptance_threshold = 0.75
                        trk = active_trackers[c]

                        # Adjust threshold based on tracker state
                        if trk.consecutive_occlusion_frames > 0:
                            acceptance_threshold = 0.85  # More strict during occlusion recovery
                        elif trk.tracking_state == 'RECOVERING':
                            acceptance_threshold = 0.70  # More lenient for recently recovered

                        if cost_matrix[r, c] < acceptance_threshold:
                            matches.append([unmatched_dets[r], active_trk_indices[c]])
                            print(f"✅ ID:{trk.id} matched with cost {cost_matrix[r, c]:.3f} (threshold: {acceptance_threshold:.3f})")

                newly_matched_dets = {m[0] for m in matches}
                unmatched_dets = [d for d in unmatched_dets if d not in newly_matched_dets]

        # STAGE 2: Enhanced recovery for lost trackers
        if lost_trk_indices and unmatched_dets:
            lost_trackers = [self.trackers[i] for i in lost_trk_indices]
            recovery_candidates = {}

            for trk_orig_idx, trk in zip(lost_trk_indices, lost_trackers):
                trk.attempt_recovery(frame_idx)
                cx, cy, pw, ph = trk.get_predicted_chw_smoothed()

                for det_idx in unmatched_dets:
                    det = detections[det_idx]
                    dx1, dy1, dx2, dy2 = det[:4]
                    dw, dh = max(1.0, dx2-dx1), max(1.0, dy2-dy1)
                    dcx, dcy = dx1 + dw/2, dy1 + dh/2

                    # Enhanced spatial gating
                    nx, ny = abs(dcx - cx) / (pw + 1e-6), abs(dcy - cy) / (ph + 1e-6)
                    pos_gate = np.sqrt(nx*nx + ny*ny)

                    # More lenient spatial gating for lost trackers
                    spatial_threshold = 5.0 if trk.lost_age < 10 else 4.0
                    if pos_gate > spatial_threshold:
                        continue

                    # Enhanced feature matching for recovery
                    clip_sim = self.compute_clip_similarity_with_gallery(
                        det_clip_features[det_idx], trk.get_clip_gallery())

                    geometric_sim = 0.0
                    if GEOMETRIC_REID_AVAILABLE:
                        geometric_sim = compute_geometric_similarity_with_gallery(
                            det_geometric_features[det_idx], trk.get_geometric_gallery())

                    # Enhanced recovery thresholds
                    recovery_clip_threshold = self.stage2_clip_threshold
                    recovery_geometric_threshold = self.stage2_geometric_threshold

                    # Apply identity verification for track hijack prevention
                    identity_verified = True
                    if self.track_hijack_prevention:
                        if len(trk.clip_feature_gallery) > 2:
                            # Compare with multiple historical features
                            historical_sims = [
                                torch.cosine_similarity(det_clip_features[det_idx].unsqueeze(0),
                                                       hist_feat.unsqueeze(0)).item()
                                for hist_feat in list(trk.clip_feature_gallery)[-3:]
                                if det_clip_features[det_idx] is not None
                            ]
                            if historical_sims:
                                identity_consistency = np.mean(historical_sims)
                                identity_verified = identity_consistency > 0.4

                    if not identity_verified:
                        continue

                    if (clip_sim < recovery_clip_threshold or
                        (GEOMETRIC_REID_AVAILABLE and geometric_sim < recovery_geometric_threshold)):
                        continue

                    # Enhanced recovery scoring
                    spatial_score = max(0, 1.0 - (pos_gate / spatial_threshold))
                    if GEOMETRIC_REID_AVAILABLE:
                        appearance_score = (self.stage2_clip_weight * clip_sim +
                                            self.stage2_geometric_weight * geometric_sim)
                    else:
                        appearance_score = clip_sim

                    # Time decay factor for lost tracks
                    time_decay = max(0.1, 1.0 - (trk.lost_age / trk.max_lost_age))

                    final_score = 0.4 * spatial_score + 0.5 * appearance_score + 0.1 * time_decay

                    if final_score > recovery_candidates.get(det_idx, (0.0, -1))[0]:
                        recovery_candidates[det_idx] = (final_score, trk_orig_idx)

            # Assign recovered tracks
            assigned_dets = set()
            sorted_candidates = sorted(recovery_candidates.items(), key=lambda item: item[1][0], reverse=True)

            for det_idx, (score, trk_idx) in sorted_candidates:
                if score >= 0.45 and det_idx not in assigned_dets:  # Slightly higher threshold for recovery
                    matches.append([det_idx, trk_idx])
                    unmatched_dets.remove(det_idx)
                    assigned_dets.add(det_idx)
                    print(f"🔄 ID:{self.trackers[trk_idx].id} recovered with score {score:.3f}")

        matched_trk_indices = {m[1] for m in matches}
        unmatched_trks = [i for i in range(len(self.trackers)) if i not in matched_trk_indices]

        return np.array(matches), unmatched_dets, unmatched_trks

    def process_frame(self, frame, detections, text_features, text_descriptions,
                      similarity_threshold, frame_idx):
        # Predict all trackers
        for tracker in self.trackers:
            tracker.predict()

        # Enhanced association with identity protection
        matched, unm_det, unm_trk = self.enhanced_associate(frame, detections, frame_idx)

        # Update matched trackers with enhanced occlusion handling AND identity verification
        for m in matched:
            det_idx, trk_idx = int(m[0]), int(m[1])
            det = detections[det_idx]
            trk = self.trackers[trk_idx]

            crop = frame[int(det[1]):int(det[3]), int(det[0]):int(det[2])]
            clip_feat = self.extract_clip_features(crop)
            geometric_feat = extract_geometric_features(crop) if GEOMETRIC_REID_AVAILABLE else None

            # Enhanced IoU vector computation for occlusion detection
            iou_vec = compute_iou_vector(trk.get_state()[0], detections)

            # Compute match quality with multi-modal features
            if clip_feat is not None:
                clip_sim = self.compute_clip_similarity_with_gallery(clip_feat[0], trk.get_clip_gallery())
            else:
                clip_sim = 0.0

            geometric_sim = 0.0
            if GEOMETRIC_REID_AVAILABLE and geometric_feat is not None:
                geometric_sim = compute_geometric_similarity_with_gallery(geometric_feat, trk.get_geometric_gallery())

            if GEOMETRIC_REID_AVAILABLE:
                match_quality = 0.4 * clip_sim + 0.6 * geometric_sim
            else:
                match_quality = clip_sim

            # Enhanced update with occlusion handling AND identity verification
            trk.update(det[:4],
                       clip_feature=clip_feat[0] if clip_feat is not None else None,
                       geometric_feature=geometric_feat,
                       confidence=det[4] if len(det) > 4 else 1.0,
                       iou_vector=iou_vec,
                       match_quality=match_quality,
                       now_frame_idx=frame_idx,
                       text_features=text_features,  # Pass for identity verification
                       text_descriptions=text_descriptions,
                       similarity_threshold=similarity_threshold)

        # Mark unmatched trackers as lost (with enhanced logic)
        for trk_idx in unm_trk:
            self.trackers[trk_idx].mark_lost(frame_idx)

        # Create new trackers for unmatched detections with enhanced identity verification
        for det_idx in unm_det:
            det = detections[int(det_idx)]
            crop = frame[int(det[1]):int(det[3]), int(det[0]):int(det[2])]
            clip_feat = self.extract_clip_features(crop)
            if clip_feat is None:
                continue

            sims = (clip_feat @ text_features.T).squeeze()
            if sims.dim() == 0:
                sims = sims.unsqueeze(0)
            best_score, best_idx = sims.max(), sims.argmax()

            # Enhanced threshold for new tracker creation (same as similarity threshold)
            creation_threshold = similarity_threshold  # Same as matching threshold

            if best_score.item() > creation_threshold:
                # Enhanced proximity check to prevent duplicate tracks AND identity conflicts
                too_close = False
                identity_conflict = False
                min_distance_threshold = 0.25

                for trk in self.trackers:
                    if trk.is_active() or (trk.is_ghost() and trk.lost_age < 15):
                        # Spatial proximity check
                        dist = iou_batch(np.array([det[:4]]), trk.get_state())
                        if dist > min_distance_threshold:
                            too_close = True
                            break

                        # Identity conflict check - prevent same description nearby
                        if (trk.text_desc_index == best_idx.item() and
                            trk.identity_lock and
                            dist > 0.05):  # Same identity but not too far
                            # Check if this could be the same person
                            if len(trk.clip_feature_gallery) > 0:
                                existing_similarity = torch.cosine_similarity(
                                    clip_feat[0].unsqueeze(0),
                                    trk.clip_feature_gallery[-1].unsqueeze(0)
                                ).item()
                                if existing_similarity < 0.7:  # Different person
                                    identity_conflict = True
                                    print(f"🚫 Identity conflict detected: New detection matches '{text_descriptions[best_idx]}' but different from existing ID:{trk.id}")
                                    break

                if not too_close and not identity_conflict:
                    geometric_feat = extract_geometric_features(crop) if GEOMETRIC_REID_AVAILABLE else None
                    new_tracker = EnhancedUltimateKalmanTracker(det[:4], now_frame_idx=frame_idx)
                    new_tracker.text_desc_index = best_idx.item()

                    # Enhanced initial update with identity establishment
                    new_tracker.update(det[:4],
                                       clip_feature=clip_feat[0],
                                       geometric_feature=geometric_feat,
                                       confidence=best_score.item(),
                                       now_frame_idx=frame_idx,
                                       text_features=text_features,
                                       text_descriptions=text_descriptions,
                                       similarity_threshold=similarity_threshold)

                    self.trackers.append(new_tracker)
                    print(f"🎯 New Enhanced Target! ID:{new_tracker.id} '{text_descriptions[best_idx]}' (Score: {best_score:.2f}, Threshold: {creation_threshold:.2f})")
                else:
                    if too_close:
                        print(f"🚫 New tracker rejected: Too close to existing tracker")
                    if identity_conflict:
                        print(f"🚫 New tracker rejected: Identity conflict with existing tracker")

        # Clean up old trackers
        self.trackers = [t for t in self.trackers if not t.should_delete()]
        return self.trackers

    def draw_enhanced_annotations(self, frame, text_descriptions):
        """Enhanced annotation with occlusion status visualization"""
        for t in self.trackers:
            if t.hits < self.min_hits_for_display and not t.is_ghost():
                continue

            # Enhanced ghost visualization
            if t.is_ghost():
                cx, cy, w, h = t.get_predicted_chw_smoothed()
                x1, y1, x2, y2 = int(cx-w/2), int(cy-h/2), int(cx+w/2), int(cy+h/2)
                opacity = max(0.3, 1.0 - (t.lost_age / t.max_lost_age))
                color = (128, 128, 128)

                # Enhanced dashed rectangle for ghosts
                dash_length = 8
                gap_length = 4
                for x in range(x1, x2, dash_length + gap_length):
                    cv2.line(frame, (x, y1), (min(x + dash_length, x2), y1), color, 2)
                    cv2.line(frame, (x, y2), (min(x + dash_length, x2), y2), color, 2)
                for y in range(y1, y2, dash_length + gap_length):
                    cv2.line(frame, (x1, y), (x1, min(y + dash_length, y2)), color, 2)
                    cv2.line(frame, (x2, y), (x2, min(y + dash_length, y2)), color, 2)

                # Enhanced ghost label with recovery info
                label = f"ID:{t.id} [GHOST {t.lost_age}] R:{t.occlusion_recovery_attempts}"
                cv2.putText(frame, label, (x1, max(15, y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
                continue

            # Enhanced active tracker visualization
            x1, y1, x2, y2 = map(int, t.get_state()[0])
            has_clip = len(t.clip_feature_gallery) > 0
            has_geometric = len(t.geometric_feature_gallery) > 0
            is_recovering = t.tracking_state == 'RECOVERING'
            occlusion_status = t.get_occlusion_status()
            is_identity_locked = getattr(t, 'identity_lock', False)

            # Enhanced color coding based on state, occlusion, and identity lock
            if is_recovering:
                color = (0, 255, 255)  # Yellow for recovering
            elif occlusion_status != "Clear":
                if "Heavy" in occlusion_status:
                    color = (0, 0, 255)  # Red for heavy occlusion
                elif "Partial" in occlusion_status:
                    color = (0, 165, 255)  # Orange for partial occlusion
                elif "Crowd" in occlusion_status:
                    color = (255, 0, 255)  # Magenta for crowd occlusion
                else:
                    color = (0, 200, 255)  # Light orange for general occlusion
            elif is_identity_locked and has_clip and has_geometric and t.average_confidence > 0.7:
                color = (0, 255, 0)  # Bright green for locked identity with high confidence
            elif is_identity_locked:
                color = (0, 200, 0)  # Green for locked identity
            elif has_clip and has_geometric and t.average_confidence > 0.7:
                color = (0, 255, 150)  # Light green for high confidence (not locked)
            elif has_clip and has_geometric:
                color = (0, 200, 255)  # Light blue for good tracking
            elif has_clip or has_geometric:
                color = (0, 150, 255)  # Blue for basic tracking
            else:
                color = (255, 0, 255)  # Magenta for poor tracking

            # Enhanced thickness based on tracking quality
            thickness = 3 if (occlusion_status != "Clear" or is_recovering) else 2
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

            # Enhanced label with more information
            if t.text_desc_index >= 0 and t.text_desc_index < len(text_descriptions):
                desc = text_descriptions[t.text_desc_index]
                desc_short = desc.split(' ')[0] if len(desc) > 15 else desc
            else:
                desc_short = "Unknown"

            # Main label
            label = f"ID:{t.id} {desc_short}"

            # Feature status
            clip_status = "C✓" if has_clip else "C✗"
            geometric_status = "G✓" if has_geometric else "G✗"
            feature_status = f"{clip_status}{geometric_status}"

            # Enhanced state information
            state_info = ""
            if is_recovering:
                state_info = " [RECOVERED]"
            elif occlusion_status != "Clear":
                state_info = f" [OCC:{occlusion_status}]"

            # Identity lock indicator
            identity_status = ""
            if is_identity_locked:
                identity_status = " 🔒"
            elif getattr(t, 'hits', 0) < getattr(t, 'identity_establishment_frames', 5):
                identity_status = f" 🔓({t.hits}/{getattr(t, 'identity_establishment_frames', 5)})"

            # Identity confidence indicator
            identity_indicator = ""
            if t.identity_confidence < self.identity_verification_threshold:
                identity_indicator = " ⚠️"

            # Hijacking protection status
            hijack_protection = ""
            non_target_count = getattr(t, 'non_target_rejection_count', 0)
            if non_target_count > 0:
                hijack_protection = f" 🛡️({non_target_count})"

            # Enhanced confidence label with more metrics
            conf_label = f"Conf:{t.average_confidence:.2f} ID:{t.identity_confidence:.2f}{identity_indicator} {feature_status} H:{t.hits}{state_info}{identity_status}{hijack_protection}"

            # Dynamic label width
            label_w = max(cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0][0],
                         cv2.getTextSize(conf_label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)[0][0], 300)

            # Enhanced label background
            cv2.rectangle(frame, (x1, y1-50), (x1+label_w, y1), color, -1)
            cv2.putText(frame, label, (x1+2, y1-30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
            cv2.putText(frame, conf_label, (x1+2, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)

        # Enhanced HUD with occlusion statistics and identity locks
        active_count = len([t for t in self.trackers if t.is_active()])
        ghost_count = len([t for t in self.trackers if t.is_ghost()])
        occluded_count = len([t for t in self.trackers if t.is_active() and t.get_occlusion_status() != "Clear"])
        recovering_count = len([t for t in self.trackers if t.tracking_state == 'RECOVERING'])
        locked_count = len([t for t in self.trackers if t.is_active() and getattr(t, 'identity_lock', False)])

        hud = f"ENHANCED Ultimate Tracker (Anti-Hijack) - Active:{active_count} Ghosts:{ghost_count} Occluded:{occluded_count} Locked:{locked_count}"
        cv2.putText(frame, hud, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        feature_hud = f"Features - CLIP:{len([t for t in self.trackers if t.is_active() and len(t.clip_feature_gallery) > 0])}"
        feature_hud += f" Geometric:{len([t for t in self.trackers if t.is_active() and len(t.geometric_feature_gallery) > 0])}"
        cv2.putText(frame, feature_hud, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)

        state_hud = f"States - Active:{len([t for t in self.trackers if t.tracking_state == 'ACTIVE'])}"
        state_hud += f" Recovering:{recovering_count} Lost:{len([t for t in self.trackers if t.tracking_state == 'LOST'])}"
        cv2.putText(frame, state_hud, (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180,180,180), 1)

        # Add hijacking protection stats
        hijack_hud = f"Protection - Rejections:{sum(getattr(t, 'non_target_rejection_count', 0) for t in self.trackers)}"
        hijack_hud += f" FalsePos:{sum(getattr(t, 'false_positive_count', 0) for t in self.trackers)}"
        cv2.putText(frame, hijack_hud, (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,200,0), 1)

        return frame

    def set_tracker_description(self, tracker, text_descriptions):
        """Helper method to set tracker description for display"""
        if hasattr(tracker, 'text_desc_index') and tracker.text_desc_index >= 0 and tracker.text_desc_index < len(text_descriptions):
            return text_descriptions[tracker.text_desc_index]
        return "Unknown"

    def process_video(self, video_path, output_path, conf_threshold, text_descriptions, similarity_threshold):
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"❌ Could not open video: {video_path}")
            return

        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or -1
        writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

        print("🔄 Pre-computing text features...")
        with torch.no_grad():
            text_inputs = self.clip_processor(text=text_descriptions, return_tensors="pt", padding=True).to(self.device)
            text_features = self.clip_model.get_text_features(**text_inputs)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        frame_idx = 0
        EnhancedUltimateKalmanTracker.count = 1
        feature_info = "CLIP + Geometric ReID + Advanced Occlusion" if GEOMETRIC_REID_AVAILABLE else "CLIP + Advanced Occlusion"
        print(f"🎬 ENHANCED ULTIMATE Multi-Modal Tracking | Features: {feature_info}")
        print(f"📝 Target: {text_descriptions[0]}")
        print(f"🎯 Similarity Threshold: {similarity_threshold}")
        print(f"🔒 Track Hijack Prevention: {'ON' if self.track_hijack_prevention else 'OFF'}")

        try:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                frame_idx += 1

                results = self.model.predict(frame, conf=conf_threshold, classes=[0], verbose=False)
                detections = results[0].boxes.data.cpu().numpy() if results and len(results) > 0 and results[0].boxes is not None else np.empty((0, 6))

                trackers = self.process_frame(frame, detections, text_features, text_descriptions, similarity_threshold, frame_idx)

                annotated_frame = self.draw_enhanced_annotations(frame.copy(), text_descriptions)
                writer.write(annotated_frame)

                if frame_idx % 30 == 0 or frame_idx == 1:
                    active = len([t for t in trackers if t.is_active()])
                    ghosts = len([t for t in trackers if t.is_ghost()])
                    occluded = len([t for t in trackers if t.is_active() and t.get_occlusion_status() != "Clear"])
                    print(f"Frame {frame_idx}/{total_frames if total_frames > 0 else '?'} | Active:{active} Ghosts:{ghosts} Occluded:{occluded} Total:{len(trackers)}")
        except KeyboardInterrupt:
            print("\n⏹️ Processing interrupted by user")
        except Exception as e:
            print(f"❌ Error during processing: {e}")
            import traceback
            traceback.print_exc()
        finally:
            cap.release()
            writer.release()
            print(f"✅ ENHANCED ULTIMATE tracking complete! Output saved to: {output_path}")

def main():
    MODEL_PATH = 'best.pt'
    VIDEO_URL = 'https://videos.pexels.com/video-files/853828/853828-hd_1280_720_30fps.mp4'
    VIDEO_PATH = 'test_vid.mp4'
    TEXT_DESCRIPTIONS = ["girl in black skirt with backpack"]
    SIMILARITY_THRESHOLD = 0.27
    OUTPUT_PATH = 'enhanced_ultimate_multimodal_tracking.mp4'

    print("🚀 ENHANCED ULTIMATE Multi-Modal Tracking System with YoloReID-style Anti-Hijacking")
    print("=" * 80)
    print("🔥 ENHANCED FEATURES:")
    print("   ✅ Stable ID assignment with YoloReID-style anti-hijacking")
    print("   ✅ Two-stage association (Active → Lost)")
    print("   ✅ CLIP visual-semantic matching")
    print("   ✅ XFeat+LightGlue+SuperPoint geometric ReID")
    print("   ✅ ADVANCED OCCLUSION HANDLING:")
    print("      🎯 Heavy occlusion detection & management")
    print("      🎯 Partial occlusion with ReID verification")
    print("      🎯 Crowd occlusion handling")
    print("      🎯 Motion consistency checking")
    print("      🎯 Pre-occlusion feature preservation")
    print("   ✅ YoloReID-INSPIRED ANTI-HIJACKING:")
    print("      🛡️ Identity lock mechanism after establishment")
    print("      🛡️ Text description consistency verification")
    print("      🛡️ Multi-modal identity verification")
    print("      🛡️ Non-target entity rejection")
    print("      🛡️ Identity conflict detection")
    print("      🛡️ Hijacking attempt counting & penalties")
    print("   ✅ Enhanced ghost tracker recovery")
    print("   ✅ Multi-modal feature fusion")
    print("   ✅ Identity confidence tracking")
    print("   ✅ Occlusion-aware feature updates")
    print("=" * 80)
    print("📋 Configuration:")
    print(f"   - Model: {MODEL_PATH} (fallback to yolov8n.pt)")
    print(f"   - Target: {TEXT_DESCRIPTIONS[0]}")
    print(f"   - Similarity Threshold: {SIMILARITY_THRESHOLD}")
    print(f"   - Advanced Tracking: {'✅' if ADVANCED_TRACKING_AVAILABLE else '❌'}")
    print(f"   - Geometric ReID: {'✅' if GEOMETRIC_REID_AVAILABLE else '❌'}")
    print(f"   - YoloReID Anti-Hijacking: ✅")
    print(f"   - Identity Lock System: ✅")
    print(f"   - Enhanced Occlusion Handling: ✅")

    if not os.path.exists(VIDEO_PATH):
        print("📥 Downloading demo video...")
        try:
            r = requests.get(VIDEO_URL, stream=True, timeout=60)
            r.raise_for_status()
            with open(VIDEO_PATH, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            print("✅ Demo video downloaded")
        except Exception as e:
            print(f"❌ Video download failed: {e}")
            return
    try:
        processor = EnhancedUltimateMultiModalTracker(model_path=MODEL_PATH)
        processor.process_video(
            video_path=VIDEO_PATH,
            output_path=OUTPUT_PATH,
            text_descriptions=TEXT_DESCRIPTIONS,
            similarity_threshold=SIMILARITY_THRESHOLD,
            conf_threshold=0.5
        )
        print("\n🎉 ENHANCED ULTIMATE Multi-Modal Tracking with YoloReID Anti-Hijacking completed successfully!")
        print(f"🎬 Output video: {OUTPUT_PATH}")
        print("\n📊 Key Improvements:")
        print("   🛡️ YoloReID-style identity lock prevents track hijacking")
        print("   🛡️ Text description consistency maintained throughout tracking")
        print("   🛡️ Multi-modal identity verification prevents ID switches")
        print("   🛡️ Non-target entity rejection protects legitimate targets")
        print("   👁️ Occlusion scenarios handled with specialized strategies")
        print("   🔄 Enhanced recovery with motion prediction")
        print("   📈 Improved ID consistency and stability")
        print("   🎯 Multi-modal feature fusion for robust tracking")
    except Exception as e:
        print(f"❌ Runtime error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()